# GEIGER-1911 · 5 — Running it

The plan is fixed: eight stations, a slot in the brass at each, two hundred hours, and a
stopping rule at the one that matters. This notebook runs it — twice, once in each of the
two worlds it was designed to distinguish — and reads off what comes back.

Running it twice is not a luxury. A design is only as good as what it does in the world
it is *wrong* about, and the second run is the only place that can be seen.

| | | reaches into |
|---|---|---|
| **1** | the counts | `Scattering.forward`, a Poisson draw |
| **2** | the fit | `design.profile_likelihood` |
| **3** | the answer, with its uncertainty | `core.Interval`, `core.rounding` |
| **4** | the answer if the other atom had been right | the same code, one argument changed |
| **5** | what the number rests on | `core.Verdict`, `core.Assumption`, `display.Card` |

In [ ]:
import sys

sys.path.insert(0, ".")

import math
import warnings

import numpy as np
import plotly.graph_objects as go

import scattering as S
from axiom.core import Assumption, Interval, Verdict, format_measured, log_likelihood
from axiom.design import profile_likelihood
from axiom.display import Card, Row, show

from axiom.display import enable

enable();  # every axiom result renders itself from here on

model, surface = S.model(), S.surface()
plan = S.plan_data()
SEED = 1911


def run(world, seed=SEED):
    # The experiment: expected counts through forward(), then one Poisson draw each.
    expected = surface.counts(plan, world)
    counts = np.random.default_rng(seed).poisson(expected)
    data = dict(plan)
    data["root_count"] = 2.0 * np.sqrt(counts)
    return counts, data


observed, measured = run(S.HARD_CENTRE)
print(f"{'station':>10} {'hours':>7} {'counts':>10} {'if hard centre':>16} "
      f"{'if diffuse atom':>17}")
if_diffuse = surface.counts(plan, S.DIFFUSE)
if_hard = surface.counts(plan, S.HARD_CENTRE)
for st, seen, hard, diffuse in zip(S.PLAN, observed, if_hard, if_diffuse):
    where = f"{st.theta_deg:.1f}d" + ("" if st.foil else " out")
    print(f"{where:>10} {st.hours:7.0f} {seen:10,d} {hard:16,.1f} {diffuse:17,.1f}")

## 1 · The counts

The two anchor stations and the four bank stations agree with both atoms to within
counting error — or rather, they agree with the hard centre and disagree with the diffuse
atom by amounts that a wider core would absorb, which is the whole reason they are not
where the argument is settled.

The two witness stations are where it is settled. At a hundred and fifty degrees the
diffuse atom predicts a hundred and nineteen counts in a hundred and ten hours, every one
of them background. Ninety-four thousand arrived.

In [ ]:
inn = np.array([bool(st.foil) for st in S.PLAN])
angle = np.array([st.theta_deg for st in S.PLAN])[inn]
seconds = np.array([st.seconds for st in S.PLAN])[inn]
aperture = plan["omega"][inn]
seen_rate = observed[inn] / (aperture * seconds)
error = np.sqrt(np.maximum(observed[inn], 1)) / (aperture * seconds)

fine = np.geomspace(0.5, 179.0, 300)
fig = S.figure("Two hundred hours, against what each atom said would happen",
               "scattering angle", "counts / sr / s", height=470)
fig.add_trace(go.Scatter(x=fine, y=S.rate_per_steradian(fine, S.HARD_CENTRE),
                         name="a hard positive centre", line={"color": S.HARD_COLOR, "width": 2}))
fig.add_trace(go.Scatter(x=fine, y=S.rate_per_steradian(fine, S.DIFFUSE),
                         name="charge spread through the atom",
                         line={"color": S.DIFFUSE_COLOR, "width": 2, "dash": "dash"}))
fig.add_trace(go.Scatter(x=angle, y=seen_rate, mode="markers", name="observed",
                         error_y={"type": "data", "array": error, "visible": True},
                         marker={"color": "#1b1f24", "size": 9, "symbol": "circle"}))
fig.update_yaxes(type="log", exponentformat="power", range=[-4, 10])
S.degrees_axis(fig, log=True)
fig.show()

back = observed[inn][-1]
print(f"at 150 degrees: {back:,} counts observed, {if_diffuse[inn][-1]:.0f} predicted by the")
print(f"diffuse atom -- a Poisson excess of {(back - if_diffuse[inn][-1]) / math.sqrt(back):,.0f} "
      f"standard errors. There is no version of this that is a fluctuation.")

## 2 · The fit

The decision is over. What is left is the measurement: **how large could the positive
charge be and still have produced this?**

`profile_likelihood` holds `lam` at each point of a grid, maximizes over everything else,
and reports the drop in log-likelihood. It is the honest instrument here because the
question is one-sided — notebook 1 showed there is no lower bound to find — and a profile
says so by going flat rather than by returning a symmetric interval that does not exist.

In [ ]:
grid = np.linspace(-4.0, 4.0, 41)
start = {**S.HARD_CENTRE, "lam": 0.0}

# The optimizer walks into parameter values where exp() overflows on its way to the
# optimum. Those excursions are scored 1e12 and discarded; the warning is numpy
# reporting an arithmetic fact about a point that is then thrown away.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    values, drops = profile_likelihood(model, measured, start, "lam", grid=grid)

drops = np.asarray(drops)
radius = np.array([S.radius_of(v) for v in values])

fig = S.figure("The profile: how large the positive charge could have been",
               "radius of the positive charge (fm)",
               "drop in log-likelihood", height=430)
fig.add_trace(go.Scatter(x=radius * 1e15, y=drops, line={"color": S.HARD_COLOR, "width": 3},
                         showlegend=False))
fig.add_hline(y=1.921, line={"dash": "dash", "color": S.SLIT_COLOR},
              annotation_text="95%, one-sided")
fig.add_vline(x=S.D_CLOSEST / 2 * 1e15, line={"dash": "dot", "color": S.TRUTH_COLOR},
              annotation_text="D/2")
fig.update_xaxes(type="log")
fig.update_yaxes(type="log", exponentformat="power", range=[-2, 5])
fig.show()

The profile rises through five orders of magnitude as the charge grows and is essentially
level as it shrinks. That shape *is* the answer: anything much above thirty femtometres is
excluded with overwhelming force, and below that the likelihood has almost nothing to say.

It is not perfectly level, and the reason is worth being precise about. There is a shallow
minimum, and to its left the drop climbs back to about two by the time the radius reaches
$D/2$. Two units of log-likelihood over a region where the model's predictions change by
less than a per cent is a counting fluctuation, not a measurement — and notebook 2 said so
in advance, when `fisher_information` reported the information about `lam` at the hard
centre as $10^{-7}$ against $10^{5}$ at a visible charge. **So the number to report is the
upper edge, and there is no lower edge to report.**

## 3 · The answer

In [ ]:
# A larger lam is a smaller charge, so the 95% edge on the side that carries information
# is the last grid point above the threshold *before* the minimum. Nothing beyond the
# minimum is read as a bound; the profile there is level to within a fluctuation.
lowest = int(np.argmin(drops))
last = int(np.where(drops[:lowest] > 1.921)[0][-1])
crossing = float(np.interp(1.921, [drops[last + 1], drops[last]],
                           [values[last + 1], values[last]]))
bound = S.radius_of(crossing)
floor = S.D_CLOSEST / 2.0
resolution = abs(radius[last] - radius[last + 1]) * 1e15  # what the grid can distinguish
print(f"the profile bottoms at {radius[lowest] * 1e15:.1f} fm and climbs only to "
      f"{drops[-1]:.1f} by {radius[-1] * 1e15:.1f} fm — no lower bound there\n")

interval = Interval(lower=0.0, upper=bound * 1e15, definition="wald", mass=0.95)
print("the radius of the positive charge of gold, in femtometres:")
print(f"  {interval.lower:.0f} to {format_measured(interval.upper, resolution)}"
      f"  ({interval.mass:.0%}, one-sided)")
print(f"\n  = R < {bound:.3g} m")
print(f"  the beam could not have resolved below   {floor * 1e15:.1f} fm")
print(f"  notebook 4 predicted the plan would give {32.3:.1f} fm")
print(f"  Rutherford published, in 1911            34 fm")

The lower end of that interval is zero and it means it. The experiment has not measured
the nucleus; it has fenced it, and the fence sits nine per cent above the closest the beam
can get. The agreement with what notebook 4 predicted before any counts existed is the
design math doing its job.

## 4 · The same plan, in the other world

Now the run that matters more. Nothing about the apparatus changes; only which atom is
real.

In [ ]:
counterfactual, other = run(S.DIFFUSE)
print(f"{'station':>10} {'in a hard-centre world':>24} {'in a diffuse world':>20}")
for st, hard, soft in zip(S.PLAN, observed, counterfactual):
    where = f"{st.theta_deg:.1f}d" + ("" if st.foil else " out")
    print(f"{where:>10} {hard:24,d} {soft:20,d}")

wide = [i for i, st in enumerate(S.PLAN) if st.foil and st.theta_deg >= 90.0]
print(f"\nthe two witness stations saw {observed[wide].sum():,} counts in one world and "
      f"{counterfactual[wide].sum():,} in the other,")
print(f"against a measured background of {plan['omega'][-1] * plan['exposure'][wide].sum() * math.exp(S.LOG_B_TRUE):,.0f}.")

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    flat_values, flat_drops = profile_likelihood(
        model, other, S.DIFFUSE, "lam", grid=np.linspace(-11.0, -6.0, 21))

print("profiling lam in a diffuse world:")
print(f"  the largest drop anywhere on the grid is {max(flat_drops):.3g}")
print("  -- the profile is flat. That is notebook 2's dead column arriving as a")
print("  likelihood: a tail nobody saw has no measurable cut-off, so the size of the")
print("  positive charge is simply not a quantity this run estimated.")

The profile of `lam` in a diffuse world is **flat**. That is not a failure of the fit; it
is exactly what notebook 2 said would happen, when `fisher_information` at the diffuse atom
named `log_a` and `lam` as dead columns.

So in that world the experiment answers the only question it can, and answering it means
fixing the cut-off and asking about the amplitude instead: *if there were a hard positive
centre, how weakly would it have to scatter to have hidden here?* `point_charge_model` is
the same model with `lam` declared `fixed`, which is how a `ModelSpec` says a parameter is
not being inferred.

In [ ]:
point_charge = S.point_charge_model()
start = {k: v for k, v in S.DIFFUSE.items() if k != "lam"} | {"log_a": -8.0}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    amp_values, amp_drops = profile_likelihood(
        point_charge, other, start, "log_a", grid=np.linspace(-14.0, 0.0, 29))

amp_drops = np.asarray(amp_drops)
over = int(np.where(amp_drops > 1.921)[0][0])
edge = float(np.interp(1.921, amp_drops[over - 1: over + 1], amp_values[over - 1: over + 1]))
print(f"95% upper limit on the single-scattering amplitude   log_a < {edge:.2f}")
print(f"what a point charge of 79e would give                log_a = {S.LOG_A_TRUE:.2f}")
print(f"\nA hard centre would have to scatter at least "
      f"{1 / math.exp(edge - S.LOG_A_TRUE):,.0f} times more weakly than a")
print("point charge to have hidden in this run -- which is a way of saying there is none.")

That is the property worth having. In a diffuse world the experiment does not come back
empty: it comes back with a limit on the thing it was built to find, five orders of
magnitude below what a hard centre would produce, **and** with a flat profile on the
radius — which is the design saying plainly what it did not measure instead of reporting
an interval it has no right to.

## 5 · What the number rests on

Every assumption the plan leaned on was chosen because something in the design measures
it. That is what makes them assumptions rather than hopes, and it is worth writing down
which is which.

In [ ]:
assumptions = (
    Assumption(name="core_width", facet="mechanism",
               statement="the multiple-scattering core is Gaussian with the fitted width",
               challenged_by="a wide-angle excess that a wider core would also explain",
               state="satisfied",
               detail={"measured_by": "the anchor stations at 1.0 and 2.5 degrees",
                       "margin": "a 10% error would be caught with certainty"}),
    Assumption(name="background", facet="measurement",
               statement="counts with the foil out are the counts with it in, less scattering",
               challenged_by="a background that depends on the foil being there",
               state="satisfied",
               detail={"measured_by": "10 hours foil-out at the 90 degree aperture"}),
    Assumption(name="aperture", facet="measurement",
               statement="each slit reports the rate at its centre",
               challenged_by="a slit wide enough for the curvature of the law to matter",
               state="satisfied",
               detail={"largest_smearing": "0.05% at 150 degrees, a twentieth of the "
                                           "counting error"}),
    Assumption(name="single_scattering", facet="mechanism",
               statement="a wide-angle count is one close encounter, not several",
               challenged_by="a thicker foil, where two encounters would compound",
               state="asserted",
               detail={"basis": "0.4 um of gold; the design never tested this and a "
                                "thickness series would"}),
)

verdict = Verdict(
    status="downgraded",
    reason="the single-scattering assumption is asserted from the foil thickness rather "
           "than measured; every other assumption the bound leans on was measured by a "
           "station the design bought for that purpose",
    assumptions=assumptions,
    route="profile likelihood on the counts, through one forward()")
show(verdict)

In [ ]:
card = Card(title="GEIGER-1911 — the radius of the positive charge of gold",
            subtitle="200 hours, 9 stations, one slit design", status="good")
card.add("verdict", "a hard positive centre", emphasis=True)
card.add("bound on the radius", f"R < {bound * 1e15:.0f} fm  (95%, one-sided)", emphasis=True)
card.add("the beam's own floor", f"{floor * 1e15:.1f} fm — set by 7.68 MeV, not by counting")
card.add("counts at 150 degrees", f"{observed[wide][-1]:,} against a background of "
                                  f"{if_diffuse[wide][-1]:.0f}")
card.add("evidence, model believed", f"{S.weight_of_evidence(if_hard[inn], if_diffuse[inn]).sum():,.0f} nats")
card.add("evidence, model attacked",
         f"{S.surviving_evidence(angle, aperture, seconds)[0].sum():,.0f} nats")
card.add("decided at", "the first interim look, 5 hours in")
card.add("assumptions", f"{sum(a.state == 'satisfied' for a in assumptions)} measured, "
                        f"{sum(a.state == 'asserted' for a in assumptions)} asserted")
card.note = ("The bound is nine per cent above the smallest radius this beam can resolve. "
             "There is nothing left to buy with time; the next factor has to be bought "
             "with energy.")
show(card)

## What the case study says

**The question.** Is the positive charge of an atom a hard dense centre or spread through
the whole atom? Two hypotheses, and the whole of the design work came from noticing they
are one model at two values of one number — the radius of the positive charge — which
turned a debate into a measurement with a resolving power, a sensitivity curve and a
stopping rule.

**Which angles.** Four roles, and no single angle plays two of them.

| | | why |
|---|---|---|
| **1.0°, 2.5°** | anchor | the only place the core's width and the tail's amplitude can be measured; below 0.9° the slit cannot be cut narrow enough |
| **5°, 10°, 20°, 45°** | bank | where the information is, if the model of the core is believed |
| **90°, 150°** | witness | where the evidence survives the model of the core being attacked; 150° because that is past what a core thirty times too wide could reach |
| **90°, foil out** | background | nothing else measures it, and every wide-angle claim rests on it |

**What the slit should look like.** An **annular slot**, not a hole: radial half-width held
at the workshop's floor of 0.05° out to twenty degrees and opening only when the arc has
gone all the way round the beam axis — 0.2° at forty-five degrees, 0.9° at ninety, 1.8° at
a hundred and fifty. Every additional steradian is bought in azimuth, because the
scattering does not depend on azimuth and the smearing does. A round hole of the same area
at ninety degrees would have to be ±13° wide and would report a rate five per cent too high
that belongs to no angle in particular.

**How long.** A hundred and ten of the two hundred hours at a hundred and fifty degrees —
which is the opposite of what the evidence-per-hour calculation says, and is right, because
evidence-per-hour is computed inside the model that is under attack.

**What comes back.** *R < 33 fm* — the plan predicted 32 before a single count existed —
against a floor of 29.6 fm that no amount of counting can beat. Rutherford published 34 fm
in 1911 from a smaller version of the same argument.

**And if the other atom had been right.** The same two hundred hours would have returned a
flat profile on the radius and a limit on the scattering amplitude five orders of magnitude
below a point charge. A design that reports a bound in one world and an interval in the
other, and says which it is doing, is the thing worth building.